<a href="https://colab.research.google.com/github/NimrahAltafAdam/llm-matching-markets-project/blob/main/llm_matching_markets_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import os

# clone repo only if not already present
if not os.path.exists("llm-matching-markets-project"):
    !git clone https://github.com/NimrahAltafAdam/llm-matching-markets-project.git

# set base path
base_path = "llm-matching-markets-project"

# load dataset from repo
df = pd.read_csv(os.path.join(base_path, "data", "10_ic_processed.csv"))

print("shape:", df.shape)

print("\ncolumns:")
print(df.columns.tolist())

print("\nfirst row:")
print(df.iloc[0])

Cloning into 'llm-matching-markets-project'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 30 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 55.74 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (9/9), done.
shape: (100, 19)

columns:
['pref_type', 'n_man', 'n_woman', 'combined_pref_json', 'man_pref_string', 'woman_pref_string', 'men_opt', 'women_opt', 'lattice', 'random_1', 'random_3', 'random_10', 'random', 'level1_q', 'level1_a', 'level2_q', 'level2_a', 'level2n_q', 'level2n_a']

first row:
pref_type                                             impartial culture
n_man                                                                10
n_woman                                                              10
combined_pref_json    {\nM: {\nM1: [W10,W1,W3,W6,W2,W4,W9,W8,W7,W5],...
man_pref_string       10,1,3,6,2,4,9,8,7,5\n8,3,10,6,2,5,4,7,1,9\n4,..

In [8]:
row = df.iloc[0]

man_prefs = row["man_pref_string"]
woman_prefs = row["woman_pref_string"]
ground_truth = row["men_opt"]

print("=== MEN PREFERENCES ===")
print(man_prefs)

print("\n=== WOMEN PREFERENCES ===")
print(woman_prefs)

print("\n=== GROUND TRUTH MATCHING ===")
print(ground_truth)

=== MEN PREFERENCES ===
10,1,3,6,2,4,9,8,7,5
8,3,10,6,2,5,4,7,1,9
4,9,3,8,5,2,6,7,1,10
2,4,6,8,7,1,5,10,3,9
3,8,5,2,7,6,10,4,9,1
7,1,9,10,3,6,5,4,2,8
6,4,1,8,9,7,3,10,5,2
8,5,3,10,9,1,7,6,2,4
8,4,1,9,5,7,3,2,6,10
2,5,1,3,7,6,10,4,9,8

=== WOMEN PREFERENCES ===
2,8,9,10,5,7,1,4,6,3
2,7,3,1,8,9,6,10,5,4
4,10,2,6,5,8,1,9,3,7
6,2,9,1,5,8,3,4,10,7
8,4,5,3,6,7,1,10,9,2
6,9,5,1,4,2,8,3,7,10
10,9,2,1,4,8,3,7,5,6
10,2,9,4,6,7,1,8,5,3
8,6,4,3,7,10,5,9,2,1
6,4,7,5,8,9,10,2,3,1

=== GROUND TRUTH MATCHING ===
[[M1, W10],[M2, W8],[M3, W9],[M4, W6],[M5, W3],[M6, W7],[M7, W1],[M8, W5],[M9, W4],[M10, W2],]


In [10]:
# convert raw strings into structured lists

men_prefs_raw = row["man_pref_string"].split("\n")
women_prefs_raw = row["woman_pref_string"].split("\n")

men_prefs = [list(map(int, line.split(","))) for line in men_prefs_raw]
women_prefs = [list(map(int, line.split(","))) for line in women_prefs_raw]

In [11]:
# build readable preference strings

men_text = ""
for i, prefs in enumerate(men_prefs):
    men_text += f"M{i+1}: " + ", ".join([f"W{w}" for w in prefs]) + "\n"

women_text = ""
for i, prefs in enumerate(women_prefs):
    women_text += f"W{i+1}: " + ", ".join([f"M{m}" for m in prefs]) + "\n"

# final prompt
prompt = f"""
You are given a stable matching problem with 10 men (M1–M10) and 10 women (W1–W10).

Each man ranks all women in order of preference, and each woman ranks all men.

MEN PREFERENCES:
{men_text}

WOMEN PREFERENCES:
{women_text}

Your task:
Find a stable matching between men and women.

Return your answer strictly in the following JSON format:
{{
  "M1": "W?",
  "M2": "W?",
  "M3": "W?",
  "M4": "W?",
  "M5": "W?",
  "M6": "W?",
  "M7": "W?",
  "M8": "W?",
  "M9": "W?",
  "M10": "W?"
}}

Rules:
- Each man must be matched to exactly one woman
- Each woman must be matched to exactly one man
- No duplicates allowed
- Ensure the matching is stable (no blocking pairs)
"""

print(prompt)


You are given a stable matching problem with 10 men (M1–M10) and 10 women (W1–W10).

Each man ranks all women in order of preference, and each woman ranks all men.

MEN PREFERENCES:
M1: W10, W1, W3, W6, W2, W4, W9, W8, W7, W5
M2: W8, W3, W10, W6, W2, W5, W4, W7, W1, W9
M3: W4, W9, W3, W8, W5, W2, W6, W7, W1, W10
M4: W2, W4, W6, W8, W7, W1, W5, W10, W3, W9
M5: W3, W8, W5, W2, W7, W6, W10, W4, W9, W1
M6: W7, W1, W9, W10, W3, W6, W5, W4, W2, W8
M7: W6, W4, W1, W8, W9, W7, W3, W10, W5, W2
M8: W8, W5, W3, W10, W9, W1, W7, W6, W2, W4
M9: W8, W4, W1, W9, W5, W7, W3, W2, W6, W10
M10: W2, W5, W1, W3, W7, W6, W10, W4, W9, W8


WOMEN PREFERENCES:
W1: M2, M8, M9, M10, M5, M7, M1, M4, M6, M3
W2: M2, M7, M3, M1, M8, M9, M6, M10, M5, M4
W3: M4, M10, M2, M6, M5, M8, M1, M9, M3, M7
W4: M6, M2, M9, M1, M5, M8, M3, M4, M10, M7
W5: M8, M4, M5, M3, M6, M7, M1, M10, M9, M2
W6: M6, M9, M5, M1, M4, M2, M8, M3, M7, M10
W7: M10, M9, M2, M1, M4, M8, M3, M7, M5, M6
W8: M10, M2, M9, M4, M6, M7, M1, M8, M5, M3
W9:

In [16]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00


In [17]:
from getpass import getpass

GROQ_API_KEY = getpass()

··········


In [18]:
import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [20]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # fast + free
    temperature=0
)

response = llm.invoke(prompt)

print(response.content)

To find a stable matching between men and women, we can use the Gale-Shapley algorithm. This algorithm works by having the men propose to the women in order of preference, and then having the women either accept or reject the proposal based on their own preferences.

Here's a Python implementation of the Gale-Shapley algorithm:

```python
class StableMatching:
    def __init__(self, men_preferences, women_preferences):
        self.men_preferences = men_preferences
        self.women_preferences = women_preferences
        self.free_men = list(men_preferences.keys())
        self.matching = {man: None for man in men_preferences.keys()}
        self.proposed = {man: [] for man in men_preferences.keys()}

    def propose(self, man):
        if not self.proposed[man]:
            woman = self.men_preferences[man][0]
            if woman not in self.matching.values():
                self.matching[man] = woman
                self.free_men.remove(man)
                self.proposed[man].app

In [23]:
import json
import re

raw_output = response.content

# find all JSON-like blocks
matches = re.findall(r'\{[\s\S]*?\}', raw_output)

llm_matching = None

for block in reversed(matches):
    cleaned = block.replace("'", '"')
    cleaned = re.sub(r',\s*}', '}', cleaned)
    try:
        llm_matching = json.loads(cleaned)
        break
    except:
        continue

if llm_matching is not None:
    print("Extracted JSON:")
    print(llm_matching)
else:
    print("No valid JSON object found.")

Extracted JSON:
{'M1': 'W10', 'M2': 'W8', 'M3': 'W4', 'M4': 'W2', 'M5': 'W3', 'M6': 'W7', 'M7': 'W6', 'M8': 'W5', 'M9': 'W9', 'M10': 'W1'}


In [24]:
# ground truth from dataset
print("Ground truth:")
print(ground_truth)

print("\nLLM matching:")
print(llm_matching)

# quick validity check
assigned_women = list(llm_matching.values())
is_valid = len(assigned_women) == len(set(assigned_women))

print("\nIs valid one-to-one matching?", is_valid)

# compare with ground truth string for now
ground_truth_clean = ground_truth.replace(" ", "")
llm_as_pairs = ",".join([f"[{m},{w}]" for m, w in llm_matching.items()])
llm_as_pairs = "[" + llm_as_pairs + "]"

print("\nLLM formatted as pair list:")
print(llm_as_pairs)

print("\nExact match with ground truth string?",
      llm_as_pairs.replace(" ", "") == ground_truth_clean)

Ground truth:
[[M1, W10],[M2, W8],[M3, W9],[M4, W6],[M5, W3],[M6, W7],[M7, W1],[M8, W5],[M9, W4],[M10, W2],]

LLM matching:
{'M1': 'W10', 'M2': 'W8', 'M3': 'W4', 'M4': 'W2', 'M5': 'W3', 'M6': 'W7', 'M7': 'W6', 'M8': 'W5', 'M9': 'W9', 'M10': 'W1'}

Is valid one-to-one matching? True

LLM formatted as pair list:
[[M1,W10],[M2,W8],[M3,W4],[M4,W2],[M5,W3],[M6,W7],[M7,W6],[M8,W5],[M9,W9],[M10,W1]]

Exact match with ground truth string? False


In [25]:
def prefers(preference_list, a, b):
    return preference_list.index(a) < preference_list.index(b)

# convert preferences into label-based dictionaries
men_pref_dict = {f"M{i+1}": [f"W{w}" for w in prefs] for i, prefs in enumerate(men_prefs)}
women_pref_dict = {f"W{i+1}": [f"M{m}" for m in prefs] for i, prefs in enumerate(women_prefs)}

# reverse matching: woman -> man
reverse_matching = {w: m for m, w in llm_matching.items()}

blocking_pairs = []

for m, current_w in llm_matching.items():
    for w in men_pref_dict[m]:
        if w == current_w:
            break
        current_m_for_w = reverse_matching[w]
        if prefers(men_pref_dict[m], w, current_w) and prefers(women_pref_dict[w], m, current_m_for_w):
            blocking_pairs.append((m, w))

print("Blocking pairs:", blocking_pairs)
print("Stable matching?", len(blocking_pairs) == 0)

Blocking pairs: [('M9', 'W4'), ('M9', 'W1'), ('M10', 'W2')]
Stable matching? False


In [27]:
def prefers(preference_list, a, b):
    return preference_list.index(a) < preference_list.index(b)


def check_stability(llm_matching, men_prefs, women_prefs):
    # convert preferences into label-based dictionaries
    men_pref_dict = {f"M{i+1}": [f"W{w}" for w in prefs] for i, prefs in enumerate(men_prefs)}
    women_pref_dict = {f"W{i+1}": [f"M{m}" for m in prefs] for i, prefs in enumerate(women_prefs)}

    # reverse matching: woman -> man
    reverse_matching = {w: m for m, w in llm_matching.items()}

    blocking_pairs = []

    for m, current_w in llm_matching.items():
        for w in men_pref_dict[m]:
            if w == current_w:
                break
            current_m_for_w = reverse_matching[w]
            if prefers(men_pref_dict[m], w, current_w) and prefers(women_pref_dict[w], m, current_m_for_w):
                blocking_pairs.append((m, w))

    return blocking_pairs, len(blocking_pairs) == 0

In [28]:
blocking_pairs, is_stable = check_stability(llm_matching, men_prefs, women_prefs)

print("Blocking pairs:", blocking_pairs)
print("Stable matching?", is_stable)

Blocking pairs: [('M9', 'W4'), ('M9', 'W1'), ('M10', 'W2')]
Stable matching? False


In [29]:
def check_validity(llm_matching, expected_size=10):
    # must have exactly expected_size men
    if len(llm_matching) != expected_size:
        return False, "wrong number of men in output"

    # all assigned partners
    assigned_women = list(llm_matching.values())

    # each woman must be unique
    if len(set(assigned_women)) != expected_size:
        return False, "duplicate women assigned"

    # check label format roughly
    expected_men = {f"M{i}" for i in range(1, expected_size + 1)}
    expected_women = {f"W{i}" for i in range(1, expected_size + 1)}

    if set(llm_matching.keys()) != expected_men:
        return False, "men labels are incorrect or incomplete"

    if not set(assigned_women).issubset(expected_women):
        return False, "women labels are incorrect"

    return True, "valid one-to-one matching"

In [30]:
is_valid, validity_msg = check_validity(llm_matching, expected_size=10)

print("Valid matching?", is_valid)
print("Validity message:", validity_msg)

Valid matching? True
Validity message: valid one-to-one matching


In [31]:
import re

def parse_ground_truth_pairs(ground_truth_string):
    pairs = re.findall(r'\[M(\d+),\s*W(\d+)\]', ground_truth_string)
    return {f"M{m}": f"W{w}" for m, w in pairs}


def exact_match_with_ground_truth(llm_matching, ground_truth_string):
    ground_truth_dict = parse_ground_truth_pairs(ground_truth_string)
    return llm_matching == ground_truth_dict, ground_truth_dict

In [32]:
is_exact_match, ground_truth_dict = exact_match_with_ground_truth(llm_matching, ground_truth)

print("Ground truth dict:")
print(ground_truth_dict)

print("\nExact match with ground truth?", is_exact_match)

Ground truth dict:
{'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}

Exact match with ground truth? False


In [33]:
import re

def evaluate_matching(llm_matching, men_prefs, women_prefs, ground_truth_string, expected_size=10):
    # validity
    is_valid, validity_msg = check_validity(llm_matching, expected_size=expected_size)

    # stability
    if is_valid:
        blocking_pairs, is_stable = check_stability(llm_matching, men_prefs, women_prefs)
    else:
        blocking_pairs, is_stable = [], False

    # ground truth comparison
    is_exact_match, ground_truth_dict = exact_match_with_ground_truth(llm_matching, ground_truth_string)

    return {
        "is_valid": is_valid,
        "validity_msg": validity_msg,
        "blocking_pairs": blocking_pairs,
        "is_stable": is_stable,
        "is_exact_match": is_exact_match,
        "ground_truth_dict": ground_truth_dict
    }

In [34]:
result = evaluate_matching(llm_matching, men_prefs, women_prefs, ground_truth, expected_size=10)

print(result)

{'is_valid': True, 'validity_msg': 'valid one-to-one matching', 'blocking_pairs': [('M9', 'W4'), ('M9', 'W1'), ('M10', 'W2')], 'is_stable': False, 'is_exact_match': False, 'ground_truth_dict': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}}


In [35]:
import json
import re

def extract_matching_from_response(raw_output):
    matches = re.findall(r'\{[\s\S]*?\}', raw_output)

    for block in reversed(matches):
        cleaned = block.replace("'", '"')
        cleaned = re.sub(r',\s*}', '}', cleaned)

        try:
            parsed = json.loads(cleaned)
            return parsed, True, "parsed successfully"
        except:
            continue

    return None, False, "no valid JSON object found"

In [36]:
parsed_matching, parsed_ok, parse_msg = extract_matching_from_response(response.content)

print("Parsed OK?", parsed_ok)
print("Parse message:", parse_msg)
print("Parsed matching:")
print(parsed_matching)

Parsed OK? True
Parse message: parsed successfully
Parsed matching:
{'M1': 'W10', 'M2': 'W8', 'M3': 'W4', 'M4': 'W2', 'M5': 'W3', 'M6': 'W7', 'M7': 'W6', 'M8': 'W5', 'M9': 'W9', 'M10': 'W1'}


In [37]:
def process_model_response(raw_output, men_prefs, women_prefs, ground_truth_string, expected_size=10):
    parsed_matching, parsed_ok, parse_msg = extract_matching_from_response(raw_output)

    if not parsed_ok:
        return {
            "parsed_ok": False,
            "parse_msg": parse_msg,
            "parsed_matching": None,
            "evaluation": None
        }

    evaluation = evaluate_matching(
        parsed_matching,
        men_prefs,
        women_prefs,
        ground_truth_string,
        expected_size=expected_size
    )

    return {
        "parsed_ok": True,
        "parse_msg": parse_msg,
        "parsed_matching": parsed_matching,
        "evaluation": evaluation
    }

In [38]:
final_result = process_model_response(
    response.content,
    men_prefs,
    women_prefs,
    ground_truth,
    expected_size=10
)

print(final_result)

{'parsed_ok': True, 'parse_msg': 'parsed successfully', 'parsed_matching': {'M1': 'W10', 'M2': 'W8', 'M3': 'W4', 'M4': 'W2', 'M5': 'W3', 'M6': 'W7', 'M7': 'W6', 'M8': 'W5', 'M9': 'W9', 'M10': 'W1'}, 'evaluation': {'is_valid': True, 'validity_msg': 'valid one-to-one matching', 'blocking_pairs': [('M9', 'W4'), ('M9', 'W1'), ('M10', 'W2')], 'is_stable': False, 'is_exact_match': False, 'ground_truth_dict': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}}}


In [41]:
from langchain_groq import ChatGroq

llm_reasoning = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

response_reasoning = llm_reasoning.invoke(prompt)

print(response_reasoning.content)

To find a stable matching between men and women, we can use the Gale-Shapley algorithm. This algorithm is a well-known method for solving the stable marriage problem.

Here's a step-by-step application of the Gale-Shapley algorithm to the given problem:

1. Initialize all men and women as free.
2. While there are still free men:
   - Choose the first free man.
   - Have him propose to his most preferred woman who has not yet rejected him.
   - If the woman is free, she accepts the proposal and they become engaged.
   - If the woman is already engaged, she compares her current partner with the new proposer.
     - If she prefers the new proposer, she accepts the proposal and becomes engaged to him, while her previous partner becomes free.
     - If she prefers her current partner, she rejects the new proposer, who becomes free.

Let's apply this algorithm to the given preferences:

Initially, all men and women are free.

1. M1 proposes to W10 (his top choice). W10 is free, so she accept

In [42]:
reasoning_result = process_model_response(
    response_reasoning.content,
    men_prefs,
    women_prefs,
    ground_truth,
    expected_size=10
)

print(reasoning_result)

{'parsed_ok': True, 'parse_msg': 'parsed successfully', 'parsed_matching': {'M1': 'W10', 'M2': 'W3', 'M3': 'W4', 'M4': 'W1', 'M5': 'W5', 'M6': 'W7', 'M7': 'W6', 'M8': 'W8', 'M9': 'W9', 'M10': 'W2'}, 'evaluation': {'is_valid': True, 'validity_msg': 'valid one-to-one matching', 'blocking_pairs': [('M2', 'W8'), ('M4', 'W6'), ('M4', 'W8'), ('M4', 'W7'), ('M9', 'W8'), ('M9', 'W4'), ('M9', 'W1')], 'is_stable': False, 'is_exact_match': False, 'ground_truth_dict': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}}}
